## Training: CLIP (image+text) multimodal classifier (v1)

This notebook fine-tunes a CLIP-based classifier for 6-way Fakeddit labels.

### Inputs
- `CLEANDFPATH`: clean CSV with `id`, text (e.g. `clean_title`), and label (e.g. `6_way_label`)
- `IMAGEDIR`: cached images folder

### Approach
- Use `AutoProcessor` to jointly tokenize text and preprocess images
- Pass through `CLIPModel`
- Fuse CLIP embeddings with simple feature interactions (abs diff + element-wise product)
- Train a small MLP classifier head

### Outputs (saved to `SAVE_DIR`)
- Best and final model weights
- Split metadata (`split_sizes.json`)
- Test predictions + config JSON


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip -q install -U transformers accelerate scikit-learn

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import AutoProcessor, CLIPModel

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 1) Setup + paths + split

We locate the clean CSV and image folder, drop rows with missing images, then create a stratified train/val/test split.
Split sizes + chosen column names are saved to disk for reproducibility.


In [ ]:
def first_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return candidates[0]

CLEANDF_CANDIDATES = [
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/clean_df.csv",
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/workingcleandf.csv",
]

IMAGEDIR_CANDIDATES = [
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/working/images",
    "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/workingimages",
]

CLEANDFPATH = first_existing_path(CLEANDF_CANDIDATES)
IMAGEDIR = first_existing_path(IMAGEDIR_CANDIDATES)

SAVE_DIR = "/content/drive/MyDrive/Fake-news-detector/multimodal_only_samples/clip_multimodal"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "clip_multimodal_best.pth")
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, "clip_multimodal_final.pth")
CONFIG_PATH = os.path.join(SAVE_DIR, "clip_multimodal_config.json")
TEST_PRED_PATH = os.path.join(SAVE_DIR, "clip_multimodal_test_predictions.csv")
SPLIT_INFO_PATH = os.path.join(SAVE_DIR, "split_sizes.json")

print("CLEANDFPATH:", CLEANDFPATH, "| exists:", os.path.exists(CLEANDFPATH))
print("IMAGEDIR:", IMAGEDIR, "| exists:", os.path.exists(IMAGEDIR))
print("SAVE_DIR:", SAVE_DIR)

In [ ]:
df = pd.read_csv(CLEANDFPATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

def pick_first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

TEXTCOL = pick_first_existing(df.columns, ["clean_title", "cleantitle", "title"])
LABELCOL = pick_first_existing(df.columns, ["6_way_label", "6waylabel", "label"])
IDCOL = pick_first_existing(df.columns, ["id"])

print("TEXTCOL:", TEXTCOL)
print("LABELCOL:", LABELCOL)
print("IDCOL:", IDCOL)

assert TEXTCOL is not None, "No text column found."
assert LABELCOL is not None, "No label column found."
assert IDCOL is not None, "No id column found."

In [ ]:
def find_image_path(image_id, image_dir):
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        candidate = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(candidate):
            return candidate
    return None

df = df.copy()
df[TEXTCOL] = df[TEXTCOL].fillna("").astype(str)
df[LABELCOL] = df[LABELCOL].astype(int)
df[IDCOL] = df[IDCOL].astype(str)

df["image_path"] = df[IDCOL].apply(lambda x: find_image_path(x, IMAGEDIR))
df = df[df["image_path"].notnull()].reset_index(drop=True)

print("Filtered shape:", df.shape)
print(df[LABELCOL].value_counts().sort_index())

In [ ]:
RANDOM_STATE = 42

df_train, df_test = train_test_split(
    df,
    test_size=0.20,
    stratify=df[LABELCOL],
    random_state=RANDOM_STATE
)

df_test, df_val = train_test_split(
    df_test,
    test_size=0.50,
    stratify=df_test[LABELCOL],
    random_state=RANDOM_STATE
)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

with open(SPLIT_INFO_PATH, "w") as f:
    json.dump({
        "train_size": int(len(df_train)),
        "val_size": int(len(df_val)),
        "test_size": int(len(df_test)),
        "text_col": TEXTCOL,
        "label_col": LABELCOL,
        "id_col": IDCOL,
        "clean_df_path": CLEANDFPATH,
        "image_dir": IMAGEDIR
    }, f, indent=2)

In [ ]:
CLIP_NAME = "openai/clip-vit-base-patch32"
MAX_LEN = 77
BATCH_SIZE = 16
NUM_CLASSES = len(sorted(df[LABELCOL].unique()))

processor = AutoProcessor.from_pretrained(CLIP_NAME)

print("CLIP_NAME:", CLIP_NAME)
print("MAX_LEN:", MAX_LEN)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_CLASSES:", NUM_CLASSES)

In [ ]:
class FakedditCLIPDataset(Dataset):
    """Minimal dataset.

    We defer heavy preprocessing to the batch `collate_fn` so we can process text+images
    together using the HuggingFace CLIP processor.
    """

    def __init__(self, df, text_col, label_col, id_col):
        self.df = df.reset_index(drop=True)
        self.text_col = text_col
        self.label_col = label_col
        self.id_col = id_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "text": str(row[self.text_col]),
            "label": int(row[self.label_col]),
            "image_id": str(row[self.id_col]),
            "image_path": row["image_path"]
        }


def clip_collate_fn(batch):
    """Convert a list of samples into a single CLIP-friendly batch."""
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    image_ids = [item["image_id"] for item in batch]

    # Load PIL images; processor handles resizing/normalization.
    images = []
    for item in batch:
        img = Image.open(item["image_path"]).convert("RGB")
        images.append(img)

    enc = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    )

    # Extra fields used later for metrics/prediction dumps.
    enc["labels"] = labels
    enc["image_id"] = image_ids
    enc["text"] = texts
    return enc

In [ ]:
train_dataset = FakedditCLIPDataset(df_train, TEXTCOL, LABELCOL, IDCOL)
val_dataset = FakedditCLIPDataset(df_val, TEXTCOL, LABELCOL, IDCOL)
test_dataset = FakedditCLIPDataset(df_test, TEXTCOL, LABELCOL, IDCOL)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=clip_collate_fn
)

sample = next(iter(train_loader))
print("input_ids:", sample["input_ids"].shape)
print("attention_mask:", sample["attention_mask"].shape)
print("pixel_values:", sample["pixel_values"].shape)
print("labels:", sample["labels"].shape)

## 2) Data pipeline (CLIP processor)

CLIP expects paired text+image inputs. The `collate_fn` uses the HF `AutoProcessor` to:
- tokenize text
- resize/normalize images
- produce tensors (`input_ids`, `attention_mask`, `pixel_values`) ready for the model


In [ ]:
class CLIPMultimodalClassifier(nn.Module):
    def __init__(self, clip_name="openai/clip-vit-base-patch32", num_classes=6, dropout=0.2):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_name)
        proj_dim = self.clip.config.projection_dim

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 4, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )

        image_feat = F.normalize(outputs.image_embeds, p=2, dim=-1)
        text_feat = F.normalize(outputs.text_embeds, p=2, dim=-1)

        abs_diff = torch.abs(image_feat - text_feat)
        elem_mul = image_feat * text_feat

        fused = torch.cat([image_feat, text_feat, abs_diff, elem_mul], dim=-1)
        logits = self.classifier(fused)
        return logits

In [ ]:
def freeze_clip_backbone(model):
    for p in model.clip.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

labels_np = df_train[LABELCOL].to_numpy()
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_np),
    y=labels_np
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class weights:", class_weights)

## 3) Model + training stages

We train in two stages:
- **head_only**: freeze the CLIP backbone and train only the classification head (fast + stable)
- **full_finetune**: unfreeze everything and fine-tune with a smaller learning rate

The best checkpoint is selected using validation loss.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, verbose=True, delta=0.0, save_path=None):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            if self.save_path is not None:
                torch.save(model.state_dict(), self.save_path)
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
            if self.save_path is not None:
                torch.save(model.state_dict(), self.save_path)


In [ ]:
from tqdm.auto import tqdm

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc="Training", leave=True)

    for batch in pbar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)

In [ ]:
def validate_one_epoch(model, loader, criterion, device, log_every=50):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    start_time = time.time()
    total_batches = len(loader)

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader, start=1):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )
            loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)
            running_loss += loss.item() * labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if batch_idx % log_every == 0 or batch_idx == 1 or batch_idx == total_batches:
                elapsed = time.time() - start_time
                print(
                    f"[val] batch {batch_idx}/{total_batches} | "
                    f"loss {loss.item():.4f} | elapsed {elapsed/60:.1f}m",
                    flush=True
                )

    val_loss = running_loss / len(loader.dataset)
    val_acc = accuracy_score(all_labels, all_preds)
    return val_loss, val_acc

In [ ]:
model = CLIPMultimodalClassifier(
    clip_name=CLIP_NAME,
    num_classes=NUM_CLASSES,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

history = {
    "epoch": [],
    "stage": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "lr": []
}

def run_stage(
    model,
    stage_name,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs,
    start_epoch=0,
    patience=3
):
    early_stopping = EarlyStopping(
        patience=patience,
        verbose=True,
        delta=0.0,
        save_path=BEST_MODEL_PATH
    )

    current_epoch = start_epoch

    for _ in range(num_epochs):
        current_epoch += 1

        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        scheduler.step(val_loss)
        early_stopping(val_loss, model)

        lr_now = optimizer.param_groups[0]["lr"]

        history["epoch"].append(current_epoch)
        history["stage"].append(stage_name)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr_now)

        print(
            f"[{stage_name}] Epoch {current_epoch} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"LR: {lr_now:.6f}"
        )

        if early_stopping.early_stop:
            print(f"Early stopping triggered in stage: {stage_name}")
            break

    return current_epoch

HEAD_ONLY_EPOCHS = 2
FULL_FINETUNE_EPOCHS = 6

# Stage 1: head only
freeze_clip_backbone(model)
print("Trainable params after freeze:", count_trainable_params(model))

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

last_epoch = run_stage(
    model=model,
    stage_name="head_only",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=HEAD_ONLY_EPOCHS,
    start_epoch=0,
    patience=2
)

# Stage 2: full fine-tune
unfreeze_all(model)
print("Trainable params after unfreeze:", count_trainable_params(model))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

last_epoch = run_stage(
    model=model,
    stage_name="full_finetune",
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=FULL_FINETUNE_EPOCHS,
    start_epoch=last_epoch,
    patience=3
)

torch.save(model.state_dict(), FINAL_MODEL_PATH)
print("Final model saved to:", FINAL_MODEL_PATH)
print("Best model saved to:", BEST_MODEL_PATH)

In [ ]:
hist_df = pd.DataFrame(history)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
plt.plot(hist_df["epoch"], hist_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(hist_df["epoch"], hist_df["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(hist_df["epoch"], hist_df["lr"], label="lr")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model = CLIPMultimodalClassifier(
    clip_name=CLIP_NAME,
    num_classes=NUM_CLASSES,
    dropout=0.2
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.eval()

print("Loaded best checkpoint.")

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    all_image_ids, all_texts = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values
            )
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss += loss.item() * labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_image_ids.extend(batch["image_id"])
            all_texts.extend(batch["text"])

    avg_loss = running_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    results_df = pd.DataFrame({
        "image_id": all_image_ids,
        "text": all_texts,
        "true_label": all_labels,
        "pred_label": all_preds
    })

    prob_array = np.array(all_probs)
    for i in range(prob_array.shape[1]):
        results_df[f"prob_class_{i}"] = prob_array[:, i]

    return avg_loss, accuracy, precision, recall, f1, results_df, all_labels, all_preds

In [ ]:
test_loss, test_acc, test_precision, test_recall, test_f1, pred_df, all_labels, all_preds = evaluate_model(
    best_model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1: {test_f1:.4f}")

In [ ]:
print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
pred_df.to_csv(TEST_PRED_PATH, index=False)
print("Saved test predictions to:", TEST_PRED_PATH)
display(pred_df.head())

In [ ]:
config = {
    "clip_name": CLIP_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "num_classes": NUM_CLASSES,
    "text_col": TEXTCOL,
    "label_col": LABELCOL,
    "id_col": IDCOL,
    "clean_df_path": CLEANDFPATH,
    "image_dir": IMAGEDIR,
    "best_model_path": BEST_MODEL_PATH,
    "final_model_path": FINAL_MODEL_PATH,
    "test_predictions_path": TEST_PRED_PATH,
    "head_only_epochs": HEAD_ONLY_EPOCHS,
    "full_finetune_epochs": FULL_FINETUNE_EPOCHS
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

print("Saved config to:", CONFIG_PATH)